# LVLMs-Saliency — Colab Setup
Run each cell **in order**. Requires a GPU runtime (T4 or better).

**Runtime → Change runtime type → GPU**

In [ ]:
# Cell 1: Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 2: Clone repo and install dependencies
import os

if not os.path.exists('/content/LVLMs-Saliency'):
    !git clone https://github.com/qiiizhen/LVLMs-Saliency.git /content/LVLMs-Saliency
else:
    !git -C /content/LVLMs-Saliency pull

%cd /content/LVLMs-Saliency

# Install LLaVA from local source
!pip install -e src/LLaVA/ -q

# Install other dependencies
!pip install transformers==4.37.2 accelerate bitsandbytes datasets Pillow seaborn tqdm scipy -q

# Silence a version-check import error in older transformers
import pathlib
vc = pathlib.Path('src/LLaVA/llava/utils/dependency_versions_check.py')
if vc.exists():
    vc.write_text('def check_dependencies(): pass\ndef check_dep(*a, **kw): pass\n')

# Comment out MPT imports that pull unavailable packages
init_path = 'src/LLaVA/llava/model/__init__.py'
with open(init_path) as f:
    init_code = f.read()
for mpt_line in [
    'from .language_model.llava_mpt import LlavaMPTForCausalLM, LlavaMPTConfig',
    'from .language_model.mpt import MPTForCausalLM, MPTConfig',
]:
    if mpt_line in init_code:
        init_code = init_code.replace(mpt_line, '# ' + mpt_line)
with open(init_path, 'w') as f:
    f.write(init_code)

# Allow AutoConfig to re-register llava without error
cfg_path = 'src/LLaVA/llava/model/language_model/llava_llama.py'
with open(cfg_path) as f:
    cfg_code = f.read()
cfg_code = cfg_code.replace(
    'AutoConfig.register("llava", LlavaConfig)',
    'AutoConfig.register("llava", LlavaConfig, exist_ok=True)'
)
with open(cfg_path, 'w') as f:
    f.write(cfg_code)

print('Done.')

In [ ]:
# Cell 3: Download LLaVA-1.5-7B model (~13 GB, takes 10-15 min)
import os
if not os.path.exists('/content/llava-v1.5-7b/config.json'):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='liuhaotian/llava-v1.5-7b',
        local_dir='/content/llava-v1.5-7b'
    )
    print('Model downloaded.')
else:
    print('Model already present.')

In [ ]:
# Cell 4: Fix CLIP encoder hardcoded path
clip_path = 'src/LLaVA/llava/model/multimodal_encoder/clip_encoder.py'
with open(clip_path) as f:
    clip_code = f.read()
clip_code = clip_code.replace(
    "self.vision_tower_name = '/root/autodl-tmp/CLIP-Vit-336'",
    'self.vision_tower_name = vision_tower'
)
with open(clip_path, 'w') as f:
    f.write(clip_code)
print('CLIP path fixed.')

# Cell 4b: Check for 499775_hall.pt (step1 output)
import os
if os.path.exists('499775_hall.pt'):
    print('499775_hall.pt found — ready.')
else:
    print('Not found — uploading:')
    from google.colab import files
    files.upload()

In [ ]:
# Cell 5: Enable 4-bit quantization (required for T4 16 GB)
with open('demo_step2_llava.py', 'r') as f:
    code = f.read()
code = code.replace('load_4bit = False', 'load_4bit = True')
with open('demo_step2_llava.py', 'w') as f:
    f.write(code)
print('4-bit quantization enabled.')

In [ ]:
# Cell 6: Patch accelerate dispatch_model for 4-bit compatibility
# Edits the source file directly so the fix survives module imports.
import accelerate.big_modeling as bm, inspect, importlib

src = inspect.getfile(bm)
with open(src) as f:
    lines = f.readlines()

patched = 0
new_lines = []
for i, line in enumerate(lines):
    if 'model.to(' in line and (i == 0 or 'try:' not in lines[i-1]):
        indent = len(line) - len(line.lstrip())
        sp = ' ' * indent
        i4 = ' ' * (indent + 4)
        new_lines += [
            f'{sp}try:\n',
            f'{i4}{line.lstrip()}',
            f'{sp}except ValueError as _e:\n',
            f'{i4}if "not supported" not in str(_e): raise\n',
        ]
        patched += 1
    else:
        new_lines.append(line)

if patched:
    with open(src, 'w') as f:
        f.writelines(new_lines)
    importlib.reload(bm)
    import transformers.modeling_utils as _mu
    _mu.dispatch_model = bm.dispatch_model
    print(f'Patched {patched} line(s) in accelerate. Ready.')
else:
    print('Nothing to patch (already patched or pattern changed).')

In [ ]:
# Cell 7: Run step2 — generates saliency map for sample 499775 (hall mode)
# PYTHONPATH is used instead of sys.path because !python starts a fresh subprocess
import os
os.environ['PYTHONPATH'] = 'src/transformers/src:src/LLaVA:' + os.environ.get('PYTHONPATH', '')

!PYTHONPATH=src/transformers/src:src/LLaVA:$PYTHONPATH \
  python demo_step2_llava.py \
    --model-path /content/llava-v1.5-7b

In [ ]:
# Cell 8: View saliency map output
from IPython.display import Image as IPImage, display
import glob, os

for f in sorted(glob.glob('onlytext_hall_499775*.png')):
    print(f)
    display(IPImage(f))

In [ ]:
# Cell 9: Download generated files to your computer
from google.colab import files
import glob

for f in glob.glob('onlytext_hall_499775*.png') + glob.glob('npy/onlytext_hall_499775*.npy'):
    files.download(f)
    print('Downloaded:', f)

# Analysis 3: Layer-wise Saliency
Run Cells 10–12 to generate per-layer npy files for sample 499775 (hall mode).
This re-runs the same pipeline but saves each of the 32 transformer layers separately.

In [ ]:
# Cell 10: Pull latest code (adds --save-layers flag)
!git -C /content/LVLMs-Saliency pull
print('Done.')

In [ ]:
# Cell 11: Re-run step2 with --save-layers for sample 499775 (hall mode)
# Saves 32 per-layer npy files: npy/onlytext_hall_499775_layer{00-31}_saliency.npy
import os
os.environ['PYTHONPATH'] = 'src/transformers/src:src/LLaVA:' + os.environ.get('PYTHONPATH', '')

!PYTHONPATH=src/transformers/src:src/LLaVA:$PYTHONPATH \
  python demo_step2_llava.py \
    --model-path /content/llava-v1.5-7b \
    --save-layers

import glob
layer_files = glob.glob('npy/onlytext_hall_499775_layer*.npy')
print(f'Generated {len(layer_files)} per-layer npy files')

In [ ]:
# Cell 12: Download per-layer npy files to your computer
# Then upload to server: scp npy/onlytext_hall_499775_layer*.npy \
#   zhenqi@longleaf.unc.edu:/work/users/z/h/zhenqi/LVLMs-Saliency/npy/
from google.colab import files
import glob

layer_files = sorted(glob.glob('npy/onlytext_hall_499775_layer*.npy'))
print(f'Downloading {len(layer_files)} files...')
for f in layer_files:
    files.download(f)
print('Done. Upload to server npy/ directory, then run analyze_layer.py')